# AI Engine API Endpoints Test
Tests for `genai-engine` API endpoints running at `http://localhost:8000`.

In [ ]:
import os
import time
import json
import requests
import boto3

BASE_URL = "http://localhost:8000"
API_PREFIX = f"{BASE_URL}/engine/v1"
KB_ID = "notebook-test-001"
DOC_ID = "doc-notebook-001"
TENANT_ID = "tenant-test-01"

def print_json(data):
    print(json.dumps(data, indent=2, ensure_ascii=False))

## 1. Health Check

In [ ]:
r = requests.get(f"{BASE_URL}/health", timeout=10)
print(f"Status: {r.status_code}")
print_json(r.json() if r.status_code == 200 else r.text)

## 2. Knowledge Base CRUD

In [ ]:
# 2a. Pre-cleanup (optional)
r = requests.delete(f"{API_PREFIX}/knowledge-bases/{KB_ID}", timeout=10)
print(f"DELETE Status: {r.status_code}")
try:
    print_json(r.json())
except:
    pass

In [ ]:
# 2b. Create Knowledge Base
r = requests.post(
    f"{API_PREFIX}/knowledge-bases",
    json={"knowledge_base_id": KB_ID},
    timeout=30
)
print(f"POST Status: {r.status_code}")
print_json(r.json() if r.status_code == 200 else r.text)

## 3. Document Processing (text_input)

In [ ]:
raw_text = (
    "Jupyter Notebook Test: AI agents with autonomous task completion became mainstream. "
    "Google announced new models. OpenAI released o3. "
)
payload = {
    "document_id": DOC_ID,
    "knowledge_base_id": KB_ID,
    "tenant_id": TENANT_ID,
    "source_type": "text_input",
    "raw_text": raw_text,
    "chunk_size": 200,
    "chunk_overlap": 20,
}
r = requests.post(f"{API_PREFIX}/documents/process", json=payload, timeout=30)
print(f"POST Process Status: {r.status_code}")
print_json(r.json() if r.status_code == 200 else r.text)

## 4. Wait for Celery & Check Chunks (text_input)

In [ ]:
MAX_WAIT = 60
POLL_INTERVAL = 3
elapsed = 0
total_chunks = 0

while elapsed < MAX_WAIT:
    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
    r = requests.get(
        f"{API_PREFIX}/documents/{DOC_ID}/chunks",
        params={"knowledge_base_id": KB_ID},
        timeout=15
    )
    if r.status_code == 200:
        data = r.json()
        total_chunks = data.get("total", 0)
        if total_chunks > 0:
            print(f"Done! Found {total_chunks} chunks after {elapsed}s.")
            print_json(data)
            break
    print(f"...waiting, {elapsed}s elapsed, total={total_chunks}")

if total_chunks == 0:
    print("Timeout or failed to get chunks.")

## 5. Document Processing (file_upload — PDF)

Flow gồm 2 bước:
1. Upload file PDF lên MinIO để lấy `storage_path`
2. Gọi `/documents/process` với `source_type=file_upload` — Celery worker sẽ dùng Datalab OCR để trích xuất text

In [ ]:
PDF_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'AI Techcrunch 2024-2026.pdf')
PDF_DOC_ID = 'doc-pdf-techcrunch-001'
PDF_STORAGE_KEY = f'uploads/{PDF_DOC_ID}.pdf'

# Step 1: Upload PDF to MinIO
s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin123',
)
with open(PDF_PATH, 'rb') as f:
    s3.put_object(
        Bucket='smartbot-v2',
        Key=PDF_STORAGE_KEY,
        Body=f,
        ContentType='application/pdf'
    )
print(f'Uploaded to MinIO: smartbot-v2/{PDF_STORAGE_KEY}')

# Step 2: Queue document processing
payload = {
    'document_id': PDF_DOC_ID,
    'knowledge_base_id': KB_ID,
    'tenant_id': TENANT_ID,
    'source_type': 'file_upload',
    'storage_path': PDF_STORAGE_KEY,
    'mime_type': 'application/pdf',
    'chunk_size': 500,
    'chunk_overlap': 50,
}
r = requests.post(f'{API_PREFIX}/documents/process', json=payload, timeout=30)
print(f'POST Process Status: {r.status_code}')
print_json(r.json() if r.status_code == 200 else r.text)

## 6. Wait for PDF Chunks (Datalab OCR — có thể mất 1–2 phút)

In [ ]:
MAX_WAIT = 180  # PDF OCR via Datalab takes longer than text_input
POLL_INTERVAL = 5
elapsed = 0
pdf_chunks = 0

while elapsed < MAX_WAIT:
    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
    r = requests.get(
        f'{API_PREFIX}/documents/{PDF_DOC_ID}/chunks',
        params={'knowledge_base_id': KB_ID},
        timeout=15
    )
    if r.status_code == 200:
        data = r.json()
        pdf_chunks = data.get('total', 0)
        if pdf_chunks > 0:
            print(f'Done! Found {pdf_chunks} chunks after {elapsed}s.')
            for chunk in data.get('chunks', [])[:3]:
                print(f'  [{chunk["position"]}] {chunk["content"][:150]}...')
            break
    print(f'...waiting, {elapsed}s elapsed, pdf_chunks={pdf_chunks}')

if pdf_chunks == 0:
    print('Timeout or failed. Check Celery worker logs for Datalab OCR errors.')

## 7. Chat / RAG

In [ ]:
payload = {
    "message": "What is the info in Jupyter Notebook Test?",
    "knowledge_base_ids": [KB_ID],
    "top_k": 3
}
r = requests.post(f"{API_PREFIX}/chat/test", json=payload, timeout=60)
print(f"POST Chat Status: {r.status_code}")
if r.headers.get('content-type', '').startswith('application/json'):
    print_json(r.json())
else:
    print(r.text)

## 8. Delete Vectors

In [ ]:
# Delete text_input doc vectors
r = requests.delete(
    f"{API_PREFIX}/documents/{DOC_ID}/vectors",
    json={"knowledge_base_id": KB_ID},
    timeout=15
)
print(f"DELETE text_input vectors: {r.status_code}")
print_json(r.json() if r.status_code == 200 else r.text)

# Delete PDF doc vectors
r = requests.delete(
    f"{API_PREFIX}/documents/{PDF_DOC_ID}/vectors",
    json={"knowledge_base_id": KB_ID},
    timeout=15
)
print(f"DELETE PDF vectors: {r.status_code}")
print_json(r.json() if r.status_code == 200 else r.text)

## 9. Reprocess Document

In [ ]:
payload = {
    "knowledge_base_id": KB_ID,
    "markdown_storage_path": "test/path/doc.md",
}
r = requests.post(
    f"{API_PREFIX}/documents/{DOC_ID}/reprocess",
    json=payload,
    timeout=15
)
print(f"POST Reprocess Status: {r.status_code}")
print_json(r.json() if r.status_code == 200 else r.text)

## 10. Cleanup

In [ ]:
r = requests.delete(f"{API_PREFIX}/knowledge-bases/{KB_ID}", timeout=10)
print(f"DELETE KB Status: {r.status_code}")
print_json(r.json() if r.status_code == 200 else r.text)